In [ ]:
import pandas as pd
import os
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, precision_score, f1_score
import numpy as np

# --- Configuration ---
sample_submission_file = 'sample_submission (2).csv'
submission_dir = 'submissions' # Folder where model submissions were saved

print(f"Loading sample submission (assumed ground truth labels) from {sample_submission_file}...")
try:
    # Load the sample submission file
    sample_df = pd.read_csv(sample_submission_file)
    # The sample file seems to have columns 'Id' and then a sequence of 'Class<id>' values
    # This is an unusual format for ground truth.
    # Let's assume the 'Class<id>' values represent the true class labels for the corresponding IDs.
    # The 'Id' column in the sample might be redundant if the order matches the test set.
    # First, get the true labels. Drop the 'Id' column.
    y_true_labels_series = sample_df.drop(columns=['Id']).iloc[0] # Get the first (and seemingly only) row of class labels
    y_true_labels = y_true_labels_series.values # Convert to numpy array
    print(f"Extracted {len(y_true_labels)} true labels from sample submission.")

    # Load test IDs to get the number of samples and potentially map if needed.
    # The order of labels in the sample submission should ideally correspond to the order of IDs in the test set,
    # and thus the order of predictions in the generated submission files.
    test_orig_df = pd.read_csv('test (2).csv') # Need to load this again here
    test_ids = test_orig_df.iloc[:, 0].tolist() # Get the ID list
    print(f"Loaded {len(test_ids)} test IDs from test (2).csv.")

    # Check if the number of sample values matches the number of test IDs
    if len(y_true_labels) != len(test_ids):
        print(f"Error: Number of labels in sample submission ({len(y_true_labels)}) does not match number of test IDs ({len(test_ids)}).")
        print("Cannot proceed with accuracy/ROC AUC/recall/precision/F1 calculation.")
        print("Exiting.")
        exit()

    print(f"True labels shape: {y_true_labels.shape}")

except FileNotFoundError:
    print(f"Error: {sample_submission_file} not found.")
    print("Exiting.")
    exit()
except Exception as e:
    print(f"Error processing sample submission file: {e}")
    print("Exiting.")
    exit()

# --- Load and Compare Model Submissions ---
print(f"\nLoading submissions from {submission_dir}...")
model_files = [f for f in os.listdir(submission_dir) if f.endswith('.csv')]
if not model_files:
    print(f"No CSV files found in the {submission_dir} directory.")
    print("Make sure you have run the model_training.py script first.")
    print("Exiting.")
    exit()

results = {}

for file in model_files:
    file_path = os.path.join(submission_dir, file)
    print(f"\nProcessing submission: {file}")

    try:
        # Load the model's submission
        model_df = pd.read_csv(file_path)
        # Ensure it has the required columns
        if 'ID' not in model_df.columns or 'Predicted' not in model_df.columns:
             print(f"Skipping {file}: Missing 'ID' or 'Predicted' column.")
             continue

        # Check if the number of predictions matches the ground truth labels
        if len(model_df) != len(y_true_labels):
            print(f"Skipping {file}: Number of predictions ({len(model_df)}) does not match ground truth ({len(y_true_labels)}).")
            continue

        # Extract predictions
        y_pred_labels = model_df['Predicted'].values

        # Calculate metrics
        accuracy = accuracy_score(y_true_labels, y_pred_labels)

        # --- ROC AUC Calculation ---
        # As discussed before, ROC AUC typically needs *predicted probabilities/scores*.
        # The current model_training.py saves only the final predicted class label.
        # Therefore, ROC AUC cannot be calculated meaningfully from these submission files.
        # We will set ROC AUC to None for all submissions generated by the current training script.
        roc_auc = None
        print(f"  Note: ROC AUC requires predicted probabilities. Calculating for {file} is not possible with current submission format.")

        # --- Recall Calculation (Macro Average) ---
        recall_macro = recall_score(y_true_labels, y_pred_labels, average='macro', zero_division=0) # Handle cases where a class has no true positives

        # --- Precision Calculation (Macro Average) ---
        precision_macro = precision_score(y_true_labels, y_pred_labels, average='macro', zero_division=0) # Handle cases where a class has no predicted positives

        # --- F1-Score Calculation (Macro Average) ---
        f1_macro = f1_score(y_true_labels, y_pred_labels, average='macro', zero_division=0) # Harmonic mean of precision and recall

        # Store results
        results[file] = {
            'Accuracy': accuracy,
            'ROC_AUC': roc_auc,
            'Recall_Macro': recall_macro,
            'Precision_Macro': precision_macro,
            'F1_Macro': f1_macro
        }
        print(f"  Accuracy for {file}: {accuracy:.4f}")
        print(f"  ROC_AUC for {file}: {roc_auc} (Cannot calculate without probabilities)")
        print(f"  Recall (Macro) for {file}: {recall_macro:.4f}")
        print(f"  Precision (Macro) for {file}: {precision_macro:.4f}")
        print(f"  F1-Score (Macro) for {file}: {f1_macro:.4f}")


    except Exception as e:
        print(f"  An error occurred while processing {file}: {e}")
        continue

# --- Print Summary ---
if results:
    print("\n--- Accuracy, ROC AUC, Recall, Precision & F1-Score Summary ---")
    print(f"{'Model File':<35} {'Acc':<6} {'ROC':<6} {'Rec':<6} {'Prec':<6} {'F1':<6}")
    print("-" * (35 + 6*5))
    for file, metrics in results.items():
        acc = metrics['Accuracy']
        auc = metrics['ROC_AUC']
        rec = metrics['Recall_Macro']
        prec = metrics['Precision_Macro']
        f1 = metrics['F1_Macro']
        auc_str = f"{auc:.4f}" if auc is not None else f"{auc}"
        print(f"{file:<35} {acc:<6.4f} {auc_str:<6} {rec:<6.4f} {prec:<6.4f} {f1:<6.4f}")

    # Find the best model based on Accuracy (primary metric often used)
    best_model_acc_file = max(results, key=lambda k: results[k]['Accuracy'])
    best_acc = results[best_model_acc_file]['Accuracy']
    print(f"\nBest Model (by Accuracy): {best_model_acc_file}")
    print(f"  Accuracy: {best_acc:.4f}")
    print(f"  ROC AUC: {results[best_model_acc_file]['ROC_AUC']:.4f}" if results[best_model_acc_file]['ROC_AUC'] is not None else f"  ROC AUC: {results[best_model_acc_file]['ROC_AUC']} (Not calculated)")
    print(f"  Recall (Macro): {results[best_model_acc_file]['Recall_Macro']:.4f}")
    print(f"  Precision (Macro): {results[best_model_acc_file]['Precision_Macro']:.4f}")
    print(f"  F1-Score (Macro): {results[best_model_acc_file]['F1_Macro']:.4f}")

    # Find the best model based on Recall (Macro)
    best_model_rec_file = max(results, key=lambda k: results[k]['Recall_Macro'])
    best_rec = results[best_model_rec_file]['Recall_Macro']
    print(f"\nBest Model (by Recall Macro): {best_model_rec_file}")
    print(f"  Accuracy: {results[best_model_rec_file]['Accuracy']:.4f}")
    print(f"  ROC AUC: {results[best_model_rec_file]['ROC_AUC']:.4f}" if results[best_model_rec_file]['ROC_AUC'] is not None else f"  ROC AUC: {results[best_model_rec_file]['ROC_AUC']} (Not calculated)")
    print(f"  Recall (Macro): {best_rec:.4f}")
    print(f"  Precision (Macro): {results[best_model_rec_file]['Precision_Macro']:.4f}")
    print(f"  F1-Score (Macro): {results[best_model_rec_file]['F1_Macro']:.4f}")

    # Find the best model based on Precision (Macro)
    best_model_prec_file = max(results, key=lambda k: results[k]['Precision_Macro'])
    best_prec = results[best_model_prec_file]['Precision_Macro']
    print(f"\nBest Model (by Precision Macro): {best_model_prec_file}")
    print(f"  Accuracy: {results[best_model_prec_file]['Accuracy']:.4f}")
    print(f"  ROC AUC: {results[best_model_prec_file]['ROC_AUC']:.4f}" if results[best_model_prec_file]['ROC_AUC'] is not None else f"  ROC AUC: {results[best_model_prec_file]['ROC_AUC']} (Not calculated)")
    print(f"  Recall (Macro): {results[best_model_prec_file]['Recall_Macro']:.4f}")
    print(f"  Precision (Macro): {best_prec:.4f}")
    print(f"  F1-Score (Macro): {results[best_model_prec_file]['F1_Macro']:.4f}")

    # Find the best model based on F1-Score (Macro)
    best_model_f1_file = max(results, key=lambda k: results[k]['F1_Macro'])
    best_f1 = results[best_model_f1_file]['F1_Macro']
    print(f"\nBest Model (by F1-Score Macro): {best_model_f1_file}")
    print(f"  Accuracy: {results[best_model_f1_file]['Accuracy']:.4f}")
    print(f"  ROC AUC: {results[best_model_f1_file]['ROC_AUC']:.4f}" if results[best_model_f1_file]['ROC_AUC'] is not None else f"  ROC AUC: {results[best_model_f1_file]['ROC_AUC']} (Not calculated)")
    print(f"  Recall (Macro): {results[best_model_f1_file]['Recall_Macro']:.4f}")
    print(f"  Precision (Macro): {results[best_model_f1_file]['Precision_Macro']:.4f}")
    print(f"  F1-Score (Macro): {best_f1:.4f}")

    # Find the best model based on ROC AUC (if calculated for any model, though unlikely with current format)
    auc_results = {k: v for k, v in results.items() if v['ROC_AUC'] is not None}
    if auc_results:
        best_model_auc_file = max(auc_results, key=lambda k: auc_results[k]['ROC_AUC'])
        best_auc = auc_results[best_model_auc_file]['ROC_AUC']
        print(f"\nBest Model (by ROC AUC): {best_model_auc_file}")
        print(f"  Accuracy: {results[best_model_auc_file]['Accuracy']:.4f}")
        print(f"  ROC AUC: {best_auc:.4f}")
        print(f"  Recall (Macro): {results[best_model_auc_file]['Recall_Macro']:.4f}")
        print(f"  Precision (Macro): {results[best_model_auc_file]['Precision_Macro']:.4f}")
        print(f"  F1-Score (Macro): {results[best_model_auc_file]['F1_Macro']:.4f}")
    else:
        print("\nNo models had valid ROC AUC scores calculated (requires probability outputs).")

else:
    print("\nNo valid submission files were processed for comparison.")